In [1]:
import numpy as np
from tslearn.datasets import UCR_UEA_datasets
import torch
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

from uni2ts.model.moirai import MoiraiModule

# Data

In [2]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Define MoiraiEncoder

In [3]:
MODEL_PARAMS = {
    "patch_size": 16,  # 8, 16, 32, 64, 128
    "num_samples": 100,
    "target_dim": 1,
    "feat_dynamic_real_dim": 0,
    "past_feat_dynamic_real_dim": 0,
}
SIZE = "small"  # model size: choose from {'small', 'base', 'large'}
device = "cuda"

In [4]:
from moirai_classification.encoder import MoiraiEncoder

encoder = MoiraiEncoder(
    module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
    prediction_length=0,
    context_length=36,
    **MODEL_PARAMS,
)

# Prepare data

Preprocess data

In [5]:
def preprocess_data(
    data: np.ndarray,
    *,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
):
    """
    data: np.ndarray of shape (N, T, V) = (n_individual, time, variate)
    Assumes NO missing values and NO padding in the raw data.
    """

    if not isinstance(data, np.ndarray):
        raise TypeError("data must be a numpy.ndarray")
    if data.ndim != 3:
        raise ValueError(f"Expected shape (N,T,V), got {data.shape}")

    N, T, V = data.shape

    # (N,T,V)
    past_target = torch.as_tensor(data, dtype=dtype, device=device)

    # observed mask: (N,T,V) all True no missing values
    past_observed_target = torch.ones((N, T, V), dtype=torch.bool, device=device)

    # padding mask: (N,T) all if no padding
    past_is_pad = torch.zeros((N, T), dtype=torch.bool, device=device)

    return past_target, past_observed_target, past_is_pad

In [6]:
X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = (
    preprocess_data(X_train, device=device)
)

X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = (
    preprocess_data(X_test, device=device)
)


print(X_train_target_tensor.shape)
print(X_train_is_target_observed.shape)
print(X_train_is_target_padded.shape)

torch.Size([2459, 36, 6])
torch.Size([2459, 36, 6])
torch.Size([2459, 36])


# Time series encoding

In [7]:
encoder.eval()
encoder.to(device)

with torch.no_grad():
    Z_train = encoder(
        past_target=X_train_target_tensor,
        past_observed_target=X_train_is_target_observed,
        past_is_pad=X_train_is_target_padded,
    )  # (N_train, combine_seq, d_model)

    Z_test = encoder(
        past_target=X_test_target_tensor,
        past_observed_target=X_test_is_target_observed,
        past_is_pad=X_test_is_target_padded,
    )  # (N_test, combine_seq, d_model)

Z_train.shape, Z_test.shape

(torch.Size([2459, 18, 384]), torch.Size([2466, 18, 384]))

Max Pooling over combine_seq dimension

In [8]:
Z_train_pooled = Z_train.mean(axis=1).cpu()
Z_test_pooled = Z_test.mean(axis=1).cpu()

Z_train_pooled.shape, Z_test_pooled.shape

(torch.Size([2459, 384]), torch.Size([2466, 384]))

# CLassification

In [9]:
from sklearn.linear_model import LogisticRegression

In [11]:
clf = LogisticRegression(max_iter=2000)
clf.fit(Z_train_pooled, y_train)
y_pred = clf.predict(Z_test_pooled)

In [12]:
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=1.0,
    )
)

              precision    recall  f1-score   support

           0       0.35      0.15      0.20       124
           1       0.87      0.93      0.90       270
           2       0.45      0.29      0.36       382
           3       0.00      0.00      0.00        63
           4       1.00      0.14      0.25         7
           5       0.55      0.17      0.26        35
           6       0.08      0.01      0.01       153
           7       0.00      0.00      0.00        24
           8       0.65      0.76      0.70       313
           9       0.00      0.00      0.00        68
          10       0.82      0.92      0.87       121
          11       0.51      0.83      0.63       777
          12       0.77      0.62      0.69        77
          13       0.60      0.17      0.27        52

    accuracy                           0.58      2466
   macro avg       0.47      0.36      0.37      2466
weighted avg       0.52      0.58      0.53      2466

